<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/tools/bocha_web_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bocha Web Search Tool

[Bocha Web Search](https://bochaai.com/) is a web search API that provides structured search results with rich metadata. This notebook demonstrates how to use the `llama-index-tools-bocha-search` package — an **external** LlamaIndex ToolSpec maintained at [github.com/NiTingKY/llama-index-tools-bocha-search](https://github.com/NiTingKY/llama-index-tools-bocha-search) — to give agents access to live web search.

The package exposes a single tool:

- **`web_search`** — searches the web via the Bocha API and returns LlamaIndex `Document` objects, each containing the result text and source metadata (`title`, `url`, `site_name`, `published_date`).

## Prerequisites

- A Bocha API key — sign up at [bochaai.com](https://bochaai.com/) to obtain one.
- An LLM API key (the examples below use OpenAI).

## Install Dependencies

In [ ]:
%pip install -q llama-index-tools-bocha-search llama-index-core llama-index-llms-openai

## Setup

Export your API keys as environment variables before running the cells below:

```bash
export BOCHA_SEARCH_API_KEY="your-bocha-api-key"
export OPENAI_API_KEY="your-openai-api-key"
```

On Windows PowerShell:

```powershell
$env:BOCHA_SEARCH_API_KEY = "your-bocha-api-key"
$env:OPENAI_API_KEY       = "your-openai-api-key"
```

In [ ]:
import os

from llama_index.tools.bocha_search import BochaSearchToolSpec

# BochaSearchToolSpec reads BOCHA_SEARCH_API_KEY from the environment by default.
# You can also pass the key directly: BochaSearchToolSpec(api_key="sk-...")
bocha_spec = BochaSearchToolSpec(
    api_key=os.getenv("BOCHA_SEARCH_API_KEY"),
    default_count=5,  # number of results per query
)

tools = bocha_spec.to_tool_list()
print(f"Available tools: {[t.metadata.name for t in tools]}")

## Direct Tool Usage

You can call `web_search` directly on the spec — useful for scripting or when you
want precise control without an agent in the loop.

In [ ]:
results = bocha_spec.web_search(query="LlamaIndex latest release", count=3)

for doc in results:
    print("Title       :", doc.metadata.get("title"))
    print("URL         :", doc.metadata.get("url"))
    print("Site        :", doc.metadata.get("site_name"))
    print("Published   :", doc.metadata.get("published_date"))
    print("Snippet     :", doc.text[:200])
    print("-" * 60)

## Agent Integration

Pass the tool list to a `FunctionAgent` so it can autonomously decide when and how
to call `web_search` in order to answer user questions.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY"))

agent = FunctionAgent(
    tools=tools,
    llm=llm,
    system_prompt=(
        "You are a helpful research assistant. "
        "Use the web_search tool to find up-to-date information before answering."
    ),
)

In [ ]:
response = await agent.run(
    "What are the key highlights of the most recent LlamaIndex release?"
)
print(str(response))

In [ ]:
response = await agent.run(
    "Find the current price of NVIDIA stock and briefly summarise today's market sentiment around it."
)
print(str(response))

## Optional: Override the API Endpoint

If your deployment uses a different Bocha endpoint, pass `api_url` at construction
time or set the `BOCHA_SEARCH_API_URL` environment variable:

```python
bocha_spec = BochaSearchToolSpec(
    api_key=os.getenv("BOCHA_SEARCH_API_KEY"),
    api_url="https://your-custom-endpoint/v1/web-search",
)
```

## References

- [Bocha Web Search](https://bochaai.com/)
- [llama-index-tools-bocha-search on GitHub](https://github.com/NiTingKY/llama-index-tools-bocha-search)
- [llama-index-tools-bocha-search on PyPI](https://pypi.org/project/llama-index-tools-bocha-search/)
- [LlamaIndex Tools documentation](/python/framework/module_guides/deploying/agents/tools)